# 🚀 LifeMate 2026: Chạy Planner & Safe Hash Guard trên Google Colab / VS Code

Notebook này được thiết kế để chạy trực tiếp thông qua **Google Colab Extension trong VS Code** hoặc mở trên web **Google Colab**.

### 💡 Thông tin về gói Google One AI Premium (AI One Pro) & GPU Colab:
- Gói **Google One AI Premium** cung cấp sẵn Compute Units cho Google Colab, cho phép bạn truy cập GPU tốc độ cao mà không lo hết quota 30h của Kaggle:
  - **T4 GPU** (15GB VRAM): Hoàn toàn đủ cho Qwen3-4B-Instruct với `--max-ctx 8192` và `--gpu-mem 0.88`.
  - **L4 GPU** (24GB VRAM): Tốc độ suy luận nhanh gấp ~2-3 lần T4, VRAM thoải mái cho cả Planner + Dialect Mapper.
  - **A100 GPU** (40GB/80GB VRAM): Tốc độ cực nhanh cho toàn bộ pipeline.

### ⚙️ Cách kết nối trong VS Code với Google Colab Extension:
1. Cài extension **Google Colab** từ VS Code Marketplace.
2. Mở file này trong VS Code -> Bấm vào nút chọn Kernel ở góc trên bên phải -> Chọn **Connect to Google Colab** (hoặc `Ctrl+Shift+P` -> gõ `Colab: Connect to Colab Runtime`).
3. Đăng nhập tài khoản Google (tài khoản có gói AI One Pro) -> Chọn phần cứng có GPU (**T4** hoặc **L4**).
4. Chạy tuần tự các cell bên dưới.

In [ ]:
# 1. Kiểm tra GPU và VRAM khả dụng
!nvidia-smi
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU Device: {device_name}')
    print(f'Total VRAM: {vram_gb:.2f} GB')

In [ ]:
# 2. Cài đặt các thư viện cần thiết
!pip install -q -U vllm transformers peft accelerate datasets chunkformer

In [ ]:
# 3. Clone hoặc cập nhật repository ZAIC_2026 trên nhánh planner/vuong
import os
from pathlib import Path

if not os.path.exists('ZAIC_2026'):
    !git clone https://github.com/Bamboohoccode/ZAIC_2026.git
    %cd ZAIC_2026
else:
    %cd ZAIC_2026
    !git fetch origin

!git checkout planner/vuong
!git pull origin planner/vuong
print('\nCommit mới nhất trên nhánh planner/vuong:')
!git log -n 1 --oneline

In [ ]:
# 4. Thiết lập đường dẫn dữ liệu data/ (Tự động nhận diện Google Drive hoặc file zip)
import os, shutil
from pathlib import Path

os.makedirs('data', exist_ok=True)

# Thử mount Google Drive nếu đang ở môi trường Colab
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except Exception as e:
    print('Bỏ qua mount Drive (đang chạy local hoặc VS Code direct):', e)

# Tự động quét tìm train.jsonl trên Drive hoặc thư mục /content
search_roots = [Path('/content/drive/MyDrive'), Path('/content'), Path('..')]
found_train = []
for root in search_roots:
    if root.exists():
        for p in root.rglob('train.jsonl'):
            if 'ZAIC_2026/data' not in str(p):
                found_train.append(p)

if found_train:
    src_train_dir = found_train[0].parent
    print(f'✅ Đã tìm thấy dữ liệu tại: {src_train_dir}')
    if not os.path.exists('data/train'):
        try:
            os.symlink(str(src_train_dir), 'data/train')
        except OSError:
            shutil.copytree(str(src_train_dir), 'data/train')
    pub_dirs = list(src_train_dir.parent.rglob('public_test.jsonl'))
    if pub_dirs and not os.path.exists('data/public_test'):
        try:
            os.symlink(str(pub_dirs[0].parent), 'data/public_test')
        except OSError:
            shutil.copytree(str(pub_dirs[0].parent), 'data/public_test')
    print('Đã liên kết dữ liệu vào data/ thành công:')
    !ls -l data/train | head -n 5
else:
    print('⚠️ CHƯA TÌM THẤY DỮ LIỆU train.jsonl!')
    print('Gợi ý xử lý:')
    print('1. Nếu bạn có folder data/ trên máy: nén thành data.zip, upload lên Google Drive (MyDrive) rồi chạy lại cell này.')
    print('2. Hoặc kéo thả data.zip vào tab Files bên trái và chạy: !unzip -q /content/data.zip -d .')

In [ ]:
# 5. Chạy Champion planner_v4 kèm Safe Hash Guard trên 300 mẫu Dev
# Pipeline đã tích hợp tự động Safe Hash Guard, tự cắt tỉa context chống tràn vLLM, và tự động lưu log ra file <out>.log
import os, torch
os.makedirs('runs', exist_ok=True)

# Tự động tối ưu gpu-mem theo VRAM thực tế:
vram = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 16
gpu_mem = 0.88 if vram < 20 else 0.90
print(f'GPU VRAM: {vram:.1f} GB -> Cấu hình --gpu-mem {gpu_mem}')

!python -m lifemate.pipeline \
    --root data \
    --split train \
    --text \
    --llm Qwen/Qwen3-4B-Instruct-2507 \
    --mapper none \
    --gpu-mem {gpu_mem} \
    --max-ctx 8192 \
    --verify-rounds 0 \
    --prompt prompts/gepa/vuong/planner_v4.txt \
    --ids eval/splits/dev_ids.txt \
    --out runs/dev_v4_guarded.jsonl

In [ ]:
# 6. Chấm điểm và phân tích lỗi chi tiết với eval/scorer.py
!python eval/scorer.py --sub runs/dev_v4_guarded.jsonl

In [ ]:
# 7. Tự động sao lưu file kết quả và run log vào Google Drive
import os, shutil
from pathlib import Path

if os.path.exists('/content/drive/MyDrive'):
    backup_dir = Path('/content/drive/MyDrive/ZAIC_2026_runs')
    backup_dir.mkdir(parents=True, exist_ok=True)
    for fname in ['dev_v4_guarded.jsonl', 'dev_v4_guarded.jsonl.log', 'dev_v4_guarded.jsonl.trace.jsonl']:
        fpath = Path('runs') / fname
        if fpath.exists():
            shutil.copy(fpath, backup_dir / fname)
            print(f'✅ Đã lưu: {fname} -> {backup_dir}')
    print(f'Toàn bộ kết quả đã được sao lưu an toàn tại Google Drive: {backup_dir}')
else:
    print('File kết quả hiện nằm trực tiếp tại: runs/dev_v4_guarded.jsonl')

In [ ]:
# 8. (Tuỳ chọn - Trụ cột 3) Chạy thử nghiệm Dialect Mapper (mapper_v1.txt + Qwen3-0.6B)
# !python -m lifemate.pipeline \
#     --root data \
#     --split train \
#     --text \
#     --llm Qwen/Qwen3-4B-Instruct-2507 \
#     --mapper Qwen/Qwen3-0.6B \
#     --mapper-prompt prompts/gepa/vuong/mapper_v1.txt \
#     --gpu-mem 0.78 \
#     --mapper-gpu-mem 0.10 \
#     --max-ctx 8192 \
#     --verify-rounds 0 \
#     --prompt prompts/gepa/vuong/planner_v4.txt \
#     --ids eval/splits/dev_ids.txt \
#     --out runs/dev_v4_mapper_v1.jsonl
#
# !python eval/scorer.py --sub runs/dev_v4_mapper_v1.jsonl